# Nine1Eight ARC-AGI-3 — Active Object World Model scored run

This is a **run-only competition notebook**. It does not train, replay known solutions, inspect
environment source, or fabricate a score. During Kaggle's hidden scoring rerun it:

1. starts an offline Qwen 3.6 27B FP8 model through vLLM;
2. writes and registers `MyAgent` with the official ARC-AGI-3 gateway runner;
3. loads every environment exactly once through the official `Swarm`;
4. logs environment loading, connected-object detection, world/goal hypotheses, ranked move
   combinations, the selected plan, each real move, and each observed transition;
5. lets the gateway produce the real `/kaggle/working/submission.parquet`.

The visible validation run writes only the required four-column gate parquet. The same notebook also
exports `/kaggle/working/agent.py` for inspection. The included Kaggle metadata attaches the competition,
Tufa Labs' public vLLM wheelhouse, and the public `vrfai/Qwen3.6-27B-FP8` snapshot.


## 1. Install the official offline ARC runtime

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORKING = Path('/kaggle/working')
WORKING.mkdir(parents=True, exist_ok=True)
COMP_ROOT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
ARC_WHEELS = COMP_ROOT / 'arc_agi_3_wheels'
TRUE_SUBMISSION = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in {'1', 'true'}

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
    '--disable-pip-version-check', '--find-links', str(ARC_WHEELS),
    'arc-agi', 'python-dotenv',
])
print(f'[BOOT] competition_rerun={TRUE_SUBMISSION} arc_wheels={ARC_WHEELS}')


## 2. Write the complete competition agent

In [ ]:
from pathlib import Path

AGENT_SOURCE = 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nimport statistics\nimport threading\nimport time\nimport urllib.request\nfrom collections import Counter, deque\nfrom dataclasses import dataclass, field\nfrom itertools import product\nfrom typing import Any, Iterable, Optional\n\nimport numpy as np\nfrom arcengine import FrameData, GameAction, GameState\n\nfrom agents.agent import Agent\n\n\nPRINT_LOCK = threading.Lock()\nMODEL_SEMAPHORE = threading.BoundedSemaphore(\n    max(1, int(os.environ.get("ARC3_MODEL_CONCURRENCY", "12")))\n)\nMODEL_ENDPOINT = os.environ.get(\n    "ARC3_MODEL_ENDPOINT", "http://127.0.0.1:1234/v1/chat/completions"\n)\nMODEL_NAME = os.environ.get("ARC3_MODEL_NAME", "arc3-local")\nMODEL_TIMEOUT_S = float(os.environ.get("ARC3_MODEL_TIMEOUT_S", "240"))\nMAX_PLAN_LENGTH = max(1, int(os.environ.get("ARC3_MAX_PLAN_LENGTH", "8")))\n\nACTION_ALIASES = {\n    "UP": "ACTION1",\n    "DOWN": "ACTION2",\n    "LEFT": "ACTION3",\n    "RIGHT": "ACTION4",\n    "SPACE": "ACTION5",\n    "INTERACT": "ACTION5",\n    "CLICK": "ACTION6",\n    "MOUSE": "ACTION6",\n    "UNDO": "ACTION7",\n    "RESET": "RESET",\n}\nDIRECTION_DELTA = {\n    "ACTION1": (-1, 0),\n    "ACTION2": (1, 0),\n    "ACTION3": (0, -1),\n    "ACTION4": (0, 1),\n}\nINVERSE_ACTION = {\n    "ACTION1": "ACTION2",\n    "ACTION2": "ACTION1",\n    "ACTION3": "ACTION4",\n    "ACTION4": "ACTION3",\n}\nCOLOR_SYMBOLS = "0123456789ABCDEF"\n\n\ndef log(game_id: str, phase: str, message: str) -> None:\n    stamp = time.strftime("%H:%M:%S")\n    with PRINT_LOCK:\n        print(f"[{stamp}][ARC3][{game_id}][{phase}] {message}", flush=True)\n\n\ndef clip_text(value: Any, limit: int = 600) -> str:\n    text = " ".join(str(value or "").split())\n    if len(text) <= limit:\n        return text\n    return text[:limit].rstrip() + f" ...[{len(text) - limit} chars omitted]"\n\n\ndef normalize_action_name(value: Any) -> str:\n    name = getattr(value, "name", value)\n    name = str(name or "").strip().upper()\n    return ACTION_ALIASES.get(name, name)\n\n\ndef grid_from_frame(frame: FrameData | Any) -> np.ndarray:\n    frames = getattr(frame, "frame", None)\n    if frames is None or len(frames) == 0:\n        return np.zeros((64, 64), dtype=np.int8)\n    grid = np.asarray(frames[-1], dtype=np.int8)\n    if grid.ndim != 2:\n        return np.zeros((64, 64), dtype=np.int8)\n    return grid\n\n\ndef grid_signature(grid: np.ndarray) -> str:\n    return hashlib.blake2s(grid.tobytes(), digest_size=8).hexdigest()\n\n\n@dataclass(frozen=True)\nclass ActionSpec:\n    name: str\n    x: Optional[int] = None\n    y: Optional[int] = None\n\n    def __post_init__(self) -> None:\n        normalized = normalize_action_name(self.name)\n        object.__setattr__(self, "name", normalized)\n        if normalized == "ACTION6":\n            if self.x is None or self.y is None:\n                raise ValueError("ACTION6 requires x and y")\n            object.__setattr__(self, "x", max(0, min(63, int(self.x))))\n            object.__setattr__(self, "y", max(0, min(63, int(self.y))))\n        else:\n            object.__setattr__(self, "x", None)\n            object.__setattr__(self, "y", None)\n\n    @property\n    def key(self) -> str:\n        if self.name == "ACTION6":\n            return f"ACTION6({self.x},{self.y})"\n        return self.name\n\n    @property\n    def data(self) -> dict[str, int]:\n        if self.name == "ACTION6":\n            return {"x": int(self.x), "y": int(self.y)}\n        return {}\n\n    def as_dict(self) -> dict[str, Any]:\n        result: dict[str, Any] = {"action": self.name}\n        if self.name == "ACTION6":\n            result.update(self.data)\n        return result\n\n\n@dataclass(frozen=True)\nclass Component:\n    object_id: int\n    color: int\n    pixels: int\n    top: int\n    left: int\n    bottom: int\n    right: int\n    row: float\n    col: float\n    shape_hash: str\n    touches_edge: bool\n    hud_like: bool\n\n    @property\n    def height(self) -> int:\n        return self.bottom - self.top + 1\n\n    @property\n    def width(self) -> int:\n        return self.right - self.left + 1\n\n    @property\n    def role_key(self) -> str:\n        return f"c{self.color}:{self.shape_hash}"\n\n    def compact(self) -> str:\n        flags = []\n        if self.touches_edge:\n            flags.append("edge")\n        if self.hud_like:\n            flags.append("hud")\n        suffix = f"/{\',\'.join(flags)}" if flags else ""\n        return (\n            f"o{self.object_id}:c{self.color} n={self.pixels} "\n            f"box=({self.top},{self.left})-({self.bottom},{self.right}) "\n            f"ctr=({self.row:.1f},{self.col:.1f}) sh={self.shape_hash}{suffix}"\n        )\n\n\n@dataclass(frozen=True)\nclass FrameView:\n    step: int\n    level: int\n    state_name: str\n    grid: np.ndarray\n    signature: str\n    background: int\n    color_counts: dict[int, int]\n    components: tuple[Component, ...]\n\n    def component_by_role(self, role_key: str) -> Optional[Component]:\n        matches = [component for component in self.components if component.role_key == role_key]\n        if not matches:\n            return None\n        return min(matches, key=lambda component: (component.hud_like, component.pixels))\n\n\n@dataclass\nclass ActionStats:\n    attempts: int = 0\n    changes: int = 0\n    noops: int = 0\n    game_overs: int = 0\n    level_completions: int = 0\n    changed_cells_total: int = 0\n    displacements: list[tuple[int, int]] = field(default_factory=list)\n\n    @property\n    def change_rate(self) -> float:\n        return (self.changes + 1.0) / (self.attempts + 2.0)\n\n    @property\n    def risk_rate(self) -> float:\n        return (self.game_overs + 0.2) / (self.attempts + 2.0)\n\n    @property\n    def progress_rate(self) -> float:\n        return (self.level_completions + 0.1) / (self.attempts + 4.0)\n\n\n@dataclass(frozen=True)\nclass TransitionEvidence:\n    step: int\n    action: ActionSpec\n    before_sig: str\n    after_sig: str\n    changed_cells: int\n    changed_bbox: Optional[tuple[int, int, int, int]]\n    color_delta: tuple[str, ...]\n    moved: tuple[str, ...]\n    level_before: int\n    level_after: int\n    state_after: str\n\n    @property\n    def level_completed(self) -> bool:\n        return self.level_after > self.level_before\n\n    def compact(self) -> str:\n        moved = ",".join(self.moved[:4]) if self.moved else "none"\n        bbox = str(self.changed_bbox) if self.changed_bbox is not None else "none"\n        delta = ",".join(self.color_delta) if self.color_delta else "none"\n        return (\n            f"t{self.step} {self.action.key} changed={self.changed_cells} "\n            f"bbox={bbox} colors={delta} moved={moved} level={self.level_before}->{self.level_after} "\n            f"state={self.state_after}"\n        )\n\n\n@dataclass(frozen=True)\nclass ComboCandidate:\n    actions: tuple[ActionSpec, ...]\n    score: float\n    rationale: str\n\n    @property\n    def key(self) -> str:\n        return ">".join(action.key for action in self.actions)\n\n    def compact(self) -> str:\n        return f"{self.key} score={self.score:.3f} because {self.rationale}"\n\n\n@dataclass(frozen=True)\nclass ModelDecision:\n    selected: tuple[ActionSpec, ...]\n    world_model: str\n    goal_model: str\n    reasoning_summary: str\n    source: str\n\n\ndef analyze_frame(frame: FrameData | Any, step: int) -> FrameView:\n    grid = grid_from_frame(frame)\n    values, counts = np.unique(grid, return_counts=True)\n    color_counts = {int(value): int(count) for value, count in zip(values, counts, strict=True)}\n    background = max(color_counts, key=color_counts.get) if color_counts else 0\n    rows, cols = grid.shape\n    visited = np.zeros(grid.shape, dtype=np.bool_)\n    components: list[Component] = []\n\n    for start_row in range(rows):\n        for start_col in range(cols):\n            if visited[start_row, start_col] or int(grid[start_row, start_col]) == background:\n                continue\n            color = int(grid[start_row, start_col])\n            stack = [(start_row, start_col)]\n            visited[start_row, start_col] = True\n            cells: list[tuple[int, int]] = []\n            while stack:\n                row, col = stack.pop()\n                cells.append((row, col))\n                for next_row, next_col in (\n                    (row - 1, col),\n                    (row + 1, col),\n                    (row, col - 1),\n                    (row, col + 1),\n                ):\n                    if not (0 <= next_row < rows and 0 <= next_col < cols):\n                        continue\n                    if visited[next_row, next_col] or int(grid[next_row, next_col]) != color:\n                        continue\n                    visited[next_row, next_col] = True\n                    stack.append((next_row, next_col))\n\n            cell_rows = [cell[0] for cell in cells]\n            cell_cols = [cell[1] for cell in cells]\n            top, bottom = min(cell_rows), max(cell_rows)\n            left, right = min(cell_cols), max(cell_cols)\n            normalized = sorted((row - top, col - left) for row, col in cells)\n            shape_bytes = json.dumps(normalized, separators=(",", ":")).encode("ascii")\n            shape_hash = hashlib.blake2s(shape_bytes, digest_size=4).hexdigest()\n            touches_edge = top == 0 or left == 0 or bottom == rows - 1 or right == cols - 1\n            height = bottom - top + 1\n            width = right - left + 1\n            long_thin = (height <= 2 and width >= max(8, cols // 3)) or (\n                width <= 2 and height >= max(8, rows // 3)\n            )\n            hud_like = touches_edge and long_thin\n            components.append(\n                Component(\n                    object_id=len(components),\n                    color=color,\n                    pixels=len(cells),\n                    top=top,\n                    left=left,\n                    bottom=bottom,\n                    right=right,\n                    row=float(sum(cell_rows)) / len(cells),\n                    col=float(sum(cell_cols)) / len(cells),\n                    shape_hash=shape_hash,\n                    touches_edge=touches_edge,\n                    hud_like=hud_like,\n                )\n            )\n\n    state_name = normalize_action_name(getattr(getattr(frame, "state", None), "name", "UNKNOWN"))\n    level = int(getattr(frame, "levels_completed", 0) or 0)\n    return FrameView(\n        step=step,\n        level=level,\n        state_name=state_name,\n        grid=grid,\n        signature=grid_signature(grid),\n        background=background,\n        color_counts=color_counts,\n        components=tuple(components),\n    )\n\n\ndef match_component_motion(before: FrameView, after: FrameView) -> list[tuple[Component, Component, int, int]]:\n    result: list[tuple[Component, Component, int, int]] = []\n    used: set[int] = set()\n    for old in before.components:\n        candidates = [\n            new\n            for new in after.components\n            if new.object_id not in used\n            and new.color == old.color\n            and new.shape_hash == old.shape_hash\n        ]\n        if not candidates:\n            continue\n        new = min(candidates, key=lambda item: abs(item.row - old.row) + abs(item.col - old.col))\n        used.add(new.object_id)\n        delta_row = int(round(new.row - old.row))\n        delta_col = int(round(new.col - old.col))\n        if delta_row != 0 or delta_col != 0:\n            result.append((old, new, delta_row, delta_col))\n    return result\n\n\ndef rle_scene(view: FrameView, max_rows: int = 64, max_runs: int = 220) -> str:\n    grid = view.grid\n    rows, cols = grid.shape\n    lines: list[str] = []\n    run_count = 0\n    for row in range(min(rows, max_rows)):\n        runs: list[str] = []\n        col = 0\n        while col < cols:\n            color = int(grid[row, col])\n            start = col\n            while col + 1 < cols and int(grid[row, col + 1]) == color:\n                col += 1\n            if color != view.background:\n                position = str(start) if start == col else f"{start}-{col}"\n                symbol = COLOR_SYMBOLS[color] if 0 <= color < len(COLOR_SYMBOLS) else str(color)\n                runs.append(f"{position}:{symbol}")\n                run_count += 1\n                if run_count >= max_runs:\n                    break\n            col += 1\n        if runs:\n            lines.append(f"r{row}:" + ",".join(runs))\n        if run_count >= max_runs:\n            lines.append("[scene truncated after exact non-background run limit]")\n            break\n    return "\\n".join(lines) if lines else "[no non-background cells]"\n\n\nclass GameWorldModel:\n    def __init__(self, game_id: str) -> None:\n        self.game_id = game_id\n        self.action_stats: dict[str, ActionStats] = {}\n        self.role_votes: Counter[str] = Counter()\n        self.combo_visits: Counter[str] = Counter()\n        self.click_visits: Counter[str] = Counter()\n        self.transitions: deque[TransitionEvidence] = deque(maxlen=24)\n        self.world_model_text = "Unknown mechanics; collecting controlled transition evidence."\n        self.goal_model_text = "Unknown objective; prefer information gain without repeating no-ops."\n        self.cross_level_notes: deque[str] = deque(maxlen=8)\n        self.last_view: Optional[FrameView] = None\n        self.last_level = 0\n\n    def _stats(self, name: str) -> ActionStats:\n        if name not in self.action_stats:\n            self.action_stats[name] = ActionStats()\n        return self.action_stats[name]\n\n    def record_transition(\n        self,\n        before: FrameView,\n        after: FrameView,\n        action: ActionSpec,\n    ) -> TransitionEvidence:\n        changed_bbox: Optional[tuple[int, int, int, int]] = None\n        color_delta: tuple[str, ...] = ()\n        if before.grid.shape == after.grid.shape:\n            changed_mask = before.grid != after.grid\n            changed_cells = int(np.count_nonzero(changed_mask))\n            if changed_cells:\n                changed_rows, changed_cols = np.nonzero(changed_mask)\n                changed_bbox = (\n                    int(changed_rows.min()),\n                    int(changed_cols.min()),\n                    int(changed_rows.max()),\n                    int(changed_cols.max()),\n                )\n                before_colors = Counter(int(value) for value in before.grid[changed_mask])\n                after_colors = Counter(int(value) for value in after.grid[changed_mask])\n                deltas = []\n                for color in sorted(set(before_colors) | set(after_colors)):\n                    delta = after_colors[color] - before_colors[color]\n                    if delta:\n                        deltas.append(f"c{color}:{delta:+d}")\n                color_delta = tuple(deltas)\n        else:\n            changed_cells = int(before.grid.size + after.grid.size)\n        motions = match_component_motion(before, after)\n        moved_descriptions = tuple(\n            f"{old.role_key}:{delta_row:+d},{delta_col:+d}"\n            for old, _new, delta_row, delta_col in motions\n        )\n        evidence = TransitionEvidence(\n            step=after.step,\n            action=action,\n            before_sig=before.signature,\n            after_sig=after.signature,\n            changed_cells=changed_cells,\n            changed_bbox=changed_bbox,\n            color_delta=color_delta,\n            moved=moved_descriptions,\n            level_before=before.level,\n            level_after=after.level,\n            state_after=after.state_name,\n        )\n        self.transitions.append(evidence)\n        stats = self._stats(action.name)\n        stats.attempts += 1\n        stats.changed_cells_total += changed_cells\n        if changed_cells:\n            stats.changes += 1\n        else:\n            stats.noops += 1\n        if after.state_name == "GAME_OVER":\n            stats.game_overs += 1\n        if evidence.level_completed:\n            stats.level_completions += 1\n\n        expected = DIRECTION_DELTA.get(action.name)\n        if expected is not None:\n            for old, _new, delta_row, delta_col in motions:\n                stats.displacements.append((delta_row, delta_col))\n                alignment = delta_row * expected[0] + delta_col * expected[1]\n                if alignment > 0:\n                    self.role_votes[old.role_key] += 2.0 + min(3.0, alignment)\n                elif alignment < 0:\n                    self.role_votes[old.role_key] -= 0.5\n\n        if action.name == "ACTION6":\n            clicked = self.component_at(before, int(action.y), int(action.x))\n            click_key = clicked.role_key if clicked is not None else f"cell:{action.x},{action.y}"\n            self.click_visits[click_key] += 1\n            if changed_cells:\n                self.role_votes[f"interactive:{click_key}"] += 1.0\n\n        if evidence.level_completed:\n            note = (\n                f"Level {before.level + 1} completed by {action.key} after "\n                f"{stats.attempts} observations of that action."\n            )\n            self.cross_level_notes.append(note)\n        self.last_view = after\n        self.last_level = after.level\n        return evidence\n\n    @staticmethod\n    def component_at(view: FrameView, row: int, col: int) -> Optional[Component]:\n        candidates = [\n            component\n            for component in view.components\n            if component.top <= row <= component.bottom and component.left <= col <= component.right\n        ]\n        if not candidates:\n            return None\n        return min(candidates, key=lambda component: component.pixels)\n\n    def player_component(self, view: FrameView) -> Optional[Component]:\n        for role_key, vote in self.role_votes.most_common():\n            if vote <= 0 or role_key.startswith("interactive:"):\n                continue\n            component = view.component_by_role(role_key)\n            if component is not None and not component.hud_like:\n                return component\n        movable = [\n            component\n            for component in view.components\n            if not component.hud_like and component.pixels <= max(64, view.grid.size // 32)\n        ]\n        if not movable:\n            return None\n        center_row = (view.grid.shape[0] - 1) / 2\n        center_col = (view.grid.shape[1] - 1) / 2\n        return min(\n            movable,\n            key=lambda component: (\n                component.touches_edge,\n                abs(component.row - center_row) + abs(component.col - center_col),\n                component.pixels,\n            ),\n        )\n\n    def likely_targets(self, view: FrameView, player: Optional[Component]) -> list[Component]:\n        candidates = [\n            component\n            for component in view.components\n            if not component.hud_like\n            and (player is None or component.object_id != player.object_id)\n            and component.pixels < max(256, view.grid.size // 8)\n        ]\n        if player is None:\n            return sorted(candidates, key=lambda component: (component.touches_edge, component.pixels))[:8]\n        return sorted(\n            candidates,\n            key=lambda component: (\n                abs(component.row - player.row) + abs(component.col - player.col),\n                component.touches_edge,\n                component.pixels,\n            ),\n        )[:8]\n\n    def direction_step(self, action_name: str) -> tuple[int, int]:\n        stats = self._stats(action_name)\n        expected = DIRECTION_DELTA.get(action_name, (0, 0))\n        aligned = [\n            displacement\n            for displacement in stats.displacements\n            if displacement[0] * expected[0] + displacement[1] * expected[1] > 0\n        ]\n        if not aligned:\n            return expected\n        return (\n            int(round(statistics.median(item[0] for item in aligned))),\n            int(round(statistics.median(item[1] for item in aligned))),\n        )\n\n    def action_stats_text(self, available: Iterable[str]) -> str:\n        parts = []\n        for name in available:\n            stats = self._stats(name)\n            parts.append(\n                f"{name}:n={stats.attempts},change={stats.change_rate:.2f},"\n                f"risk={stats.risk_rate:.2f},wins={stats.level_completions},noops={stats.noops}"\n            )\n        return "; ".join(parts)\n\n    def generate_candidates(self, view: FrameView, available: set[str]) -> list[ComboCandidate]:\n        raw: list[tuple[ActionSpec, ...]] = []\n        directions = [name for name in DIRECTION_DELTA if name in available]\n        simple = [name for name in [*directions, "ACTION5", "ACTION7"] if name in available]\n\n        for name in simple:\n            raw.append((ActionSpec(name),))\n        for name in directions:\n            for length in (2, 3, 4, 6):\n                raw.append(tuple(ActionSpec(name) for _ in range(length)))\n        for length in (2, 3):\n            for names in product(directions, repeat=length):\n                if any(INVERSE_ACTION.get(names[index]) == names[index + 1] for index in range(length - 1)):\n                    continue\n                raw.append(tuple(ActionSpec(name) for name in names))\n\n        player = self.player_component(view)\n        role_is_observed = bool(self.role_votes) and max(self.role_votes.values(), default=0.0) > 0\n        if role_is_observed and player is not None and directions:\n            for target in self.likely_targets(view, player)[:4]:\n                vertical_name = "ACTION2" if target.row > player.row else "ACTION1"\n                horizontal_name = "ACTION4" if target.col > player.col else "ACTION3"\n                vertical_step = max(1, abs(self.direction_step(vertical_name)[0]))\n                horizontal_step = max(1, abs(self.direction_step(horizontal_name)[1]))\n                vertical_count = min(MAX_PLAN_LENGTH, int(round(abs(target.row - player.row) / vertical_step)))\n                horizontal_count = min(MAX_PLAN_LENGTH, int(round(abs(target.col - player.col) / horizontal_step)))\n                vertical = [vertical_name] * vertical_count if vertical_name in available else []\n                horizontal = [horizontal_name] * horizontal_count if horizontal_name in available else []\n                for names in (vertical + horizontal, horizontal + vertical):\n                    if names:\n                        raw.append(tuple(ActionSpec(name) for name in names[:MAX_PLAN_LENGTH]))\n        if "ACTION5" in available:\n            for name in directions:\n                raw.append((ActionSpec(name), ActionSpec("ACTION5")))\n                raw.append((ActionSpec("ACTION5"), ActionSpec(name)))\n\n        if "ACTION6" in available:\n            click_components = sorted(\n                [component for component in view.components if not component.hud_like],\n                key=lambda component: (\n                    self.click_visits[component.role_key],\n                    component.touches_edge,\n                    component.pixels,\n                ),\n            )[:18]\n            for component in click_components:\n                raw.append(\n                    (\n                        ActionSpec(\n                            "ACTION6",\n                            x=int(round(component.col)),\n                            y=int(round(component.row)),\n                        ),\n                    )\n                )\n\n        if not raw:\n            raw = [(ActionSpec(name),) for name in sorted(available) if name != "RESET"]\n\n        unique: dict[str, tuple[ActionSpec, ...]] = {}\n        for combo in raw:\n            if not combo or len(combo) > MAX_PLAN_LENGTH:\n                continue\n            key = ">".join(action.key for action in combo)\n            unique.setdefault(key, combo)\n\n        scored = sorted(\n            [self.score_combo(view, combo) for combo in unique.values()],\n            key=lambda candidate: (-candidate.score, len(candidate.actions), candidate.key),\n        )\n        diverse: list[ComboCandidate] = []\n        seen_keys: set[str] = set()\n        for first_action in sorted(available - {"RESET"}):\n            representative = next(\n                (candidate for candidate in scored if candidate.actions[0].name == first_action),\n                None,\n            )\n            if representative is not None:\n                diverse.append(representative)\n                seen_keys.add(representative.key)\n        for candidate in scored:\n            if candidate.key in seen_keys:\n                continue\n            diverse.append(candidate)\n            seen_keys.add(candidate.key)\n            if len(diverse) >= 16:\n                break\n        return sorted(\n            diverse[:16],\n            key=lambda candidate: (-candidate.score, len(candidate.actions), candidate.key),\n        )\n\n    def score_combo(self, view: FrameView, combo: tuple[ActionSpec, ...]) -> ComboCandidate:\n        change_values: list[float] = []\n        risk = 0.0\n        progress_values: list[float] = []\n        evidence_attempts = 0\n        for action in combo:\n            stats = self._stats(action.name)\n            change_values.append(stats.change_rate)\n            risk += stats.risk_rate\n            progress_values.append(stats.progress_rate)\n            evidence_attempts += stats.attempts\n\n        change_value = statistics.fmean(change_values) if change_values else 0.0\n        progress_history = max(progress_values, default=0.0)\n\n        key = ">".join(action.key for action in combo)\n        novelty = 1.0 / (1.0 + self.combo_visits[key])\n        score = 0.65 * change_value + 1.8 * progress_history + 0.45 * novelty - 1.3 * risk\n        reasons = [\n            f"change={change_value:.2f}",\n            f"risk={risk:.2f}",\n            f"novelty={novelty:.2f}",\n        ]\n\n        player = self.player_component(view)\n        targets = self.likely_targets(view, player)\n        role_is_observed = bool(self.role_votes) and max(self.role_votes.values(), default=0.0) > 0\n        if (\n            role_is_observed\n            and player is not None\n            and targets\n            and all(action.name in DIRECTION_DELTA for action in combo)\n        ):\n            target = targets[0]\n            before_distance = abs(target.row - player.row) + abs(target.col - player.col)\n            predicted_row, predicted_col = player.row, player.col\n            for action in combo:\n                delta_row, delta_col = self.direction_step(action.name)\n                predicted_row += delta_row\n                predicted_col += delta_col\n            after_distance = abs(target.row - predicted_row) + abs(target.col - predicted_col)\n            reduction = before_distance - after_distance\n            confidence = min(1.0, evidence_attempts / max(1.0, len(combo)))\n            score += (0.18 + 0.37 * confidence) * max(-4.0, min(4.0, reduction))\n            reasons.append(f"target_distance_delta={reduction:.1f}")\n            rows, cols = view.grid.shape\n            if not (0 <= predicted_row < rows and 0 <= predicted_col < cols):\n                score -= 2.5\n                reasons.append("boundary_penalty")\n\n        if combo[0].name == "ACTION6":\n            component = self.component_at(view, int(combo[0].y), int(combo[0].x))\n            if component is not None:\n                attempts = self.click_visits[component.role_key]\n                score += 0.8 / (1 + attempts)\n                reasons.append(f"click_object={component.role_key}/tried={attempts}")\n            else:\n                score -= 0.6\n                reasons.append("empty_click_penalty")\n\n        reversals = sum(\n            1\n            for index in range(len(combo) - 1)\n            if INVERSE_ACTION.get(combo[index].name) == combo[index + 1].name\n        )\n        score -= 0.16 * len(combo) + 0.9 * reversals\n        deterministic_tiebreak = int(hashlib.blake2s((view.signature + key).encode(), digest_size=2).hexdigest(), 16)\n        score += deterministic_tiebreak / 65535.0 * 0.001\n        return ComboCandidate(actions=combo, score=score, rationale=", ".join(reasons))\n\n\nclass ModelProtocolError(ValueError):\n    """The configured model endpoint did not return a usable planning response."""\n\n\nclass LocalModelPlanner:\n    def __init__(self, game_id: str, world: GameWorldModel) -> None:\n        self.game_id = game_id\n        self.world = world\n        self.failures = 0\n        self.disabled = os.environ.get("ARC3_DISABLE_MODEL", "").strip().lower() in {\n            "1",\n            "true",\n            "yes",\n        }\n\n    def choose(\n        self,\n        view: FrameView,\n        candidates: list[ComboCandidate],\n        available: set[str],\n    ) -> ModelDecision:\n        if not candidates:\n            non_reset = [name for name in sorted(available) if name != "RESET"]\n            fallback_name = non_reset[0] if non_reset else "RESET"\n            return ModelDecision(\n                selected=(ActionSpec(fallback_name),),\n                world_model=self.world.world_model_text,\n                goal_model=self.world.goal_model_text,\n                reasoning_summary=f"No candidate combo was available; selected legal fallback {fallback_name}.",\n                source="heuristic",\n            )\n        if self.disabled:\n            return self._heuristic(candidates, "local model disabled by preflight or prior failures")\n\n        payload = self._payload(view, candidates, available)\n        request = urllib.request.Request(\n            MODEL_ENDPOINT,\n            data=json.dumps(payload).encode("utf-8"),\n            headers={"Content-Type": "application/json", "Authorization": "Bearer arc3-local"},\n            method="POST",\n        )\n        try:\n            with MODEL_SEMAPHORE:\n                with urllib.request.urlopen(request, timeout=MODEL_TIMEOUT_S) as response:\n                    raw_body = response.read().decode("utf-8")\n            try:\n                body = json.loads(raw_body)\n            except json.JSONDecodeError as exc:\n                raise ModelProtocolError("model response body was not valid JSON") from exc\n            content = self._extract_model_content(body)\n            parsed = self._parse_json(content)\n            decision = self._validate_decision(parsed, candidates, available)\n            self.failures = 0\n            return decision\n        except ModelProtocolError as exc:\n            self.failures += 1\n            self.disabled = True\n            log(\n                self.game_id,\n                "MODEL",\n                "protocol mismatch; disabling model planner immediately: "\n                f"{clip_text(exc, 240)}",\n            )\n            return self._heuristic(candidates, "model protocol mismatch")\n        except Exception as exc:\n            self.failures += 1\n            if self.failures >= 3:\n                self.disabled = True\n            log(self.game_id, "MODEL", f"planner failure {self.failures}: {type(exc).__name__}: {clip_text(exc, 240)}")\n            return self._heuristic(candidates, f"model failure: {type(exc).__name__}")\n\n    def _payload(\n        self,\n        view: FrameView,\n        candidates: list[ComboCandidate],\n        available: set[str],\n    ) -> dict[str, Any]:\n        player = self.world.player_component(view)\n        targets = self.world.likely_targets(view, player)\n        component_lines = [component.compact() for component in view.components[:32]]\n        transition_lines = [transition.compact() for transition in list(self.world.transitions)[-12:]]\n        candidate_lines = [candidate.compact() for candidate in candidates]\n        prompt = f"""\nGAME: {self.game_id}\nTURN: {view.step}; LEVELS_COMPLETED: {view.level}; STATE: {view.state_name}\nAVAILABLE: {sorted(available)}\nBACKGROUND: color {view.background}; PALETTE_COUNTS: {view.color_counts}\nCURRENT_WORLD_MODEL: {self.world.world_model_text}\nCURRENT_GOAL_MODEL: {self.world.goal_model_text}\nCROSS_LEVEL_MEMORY: {list(self.world.cross_level_notes)}\nACTION_EVIDENCE: {self.world.action_stats_text(sorted(available))}\nLIKELY_PLAYER: {player.compact() if player else \'unresolved\'}\nLIKELY_TARGETS: {[target.compact() for target in targets]}\nOBJECTS:\\n{chr(10).join(component_lines) if component_lines else \'[none]\'}\nRECENT_TRANSITIONS:\\n{chr(10).join(transition_lines) if transition_lines else \'[none yet]\'}\nEXACT_NONBACKGROUND_ROW_RUNS:\\n{rle_scene(view)}\nCANDIDATE_COMBOS:\\n{chr(10).join(f\'{index}. {line}\' for index, line in enumerate(candidate_lines))}\n\nSelect one listed combo. Treat every real environment action as costly. Prefer a short discriminating\nprobe while the mechanics or objective are uncertain; prefer a direct reliable plan after evidence is\nstrong. Re-ground after level changes. Do not click HUD/timer bars. Never invent an unavailable action.\nReturn only one JSON object with exactly these keys:\n{{"world_model":"concise testable mechanics summary","goal_model":"concise objective hypothesis",\n"selected_candidate":0,"reasoning_summary":"brief evidence-based reason, no hidden chain of thought"}}\n""".strip()\n        system = (\n            "You are the planning module for an ARC-AGI-3 competition agent. Frames are 64x64 color "\n            "grids. ACTION1=up, ACTION2=down, ACTION3=left, ACTION4=right, ACTION5=interact/space, "\n            "ACTION6=click x,y, ACTION7=undo. Infer object roles and goals from observed transitions. "\n            "Maintain a falsifiable world model, compare candidate move combinations, and output strict JSON."\n        )\n        return {\n            "model": MODEL_NAME,\n            "messages": [\n                {"role": "system", "content": system},\n                {"role": "user", "content": prompt},\n            ],\n            "temperature": 0.35,\n            "top_p": 0.9,\n            "max_tokens": 900,\n            "stream": False,\n        }\n\n    @staticmethod\n    def _extract_model_content(body: Any) -> str:\n        if not isinstance(body, dict):\n            raise ModelProtocolError("model response must be a JSON object")\n\n        def text_from(value: Any) -> str:\n            if isinstance(value, str):\n                return value.strip()\n            if isinstance(value, list):\n                return "\\n".join(part for item in value if (part := text_from(item))).strip()\n            if isinstance(value, dict):\n                for key in ("text", "output_text", "value", "content"):\n                    part = text_from(value.get(key))\n                    if part:\n                        return part\n            return ""\n\n        direct = text_from(body.get("output_text"))\n        if direct:\n            return direct\n\n        choices = body.get("choices")\n        if isinstance(choices, list) and choices and isinstance(choices[0], dict):\n            choice = choices[0]\n            message = choice.get("message")\n            if isinstance(message, dict):\n                content = text_from(message.get("content"))\n                if content:\n                    return content\n\n                tool_calls = message.get("tool_calls")\n                if isinstance(tool_calls, list):\n                    arguments = []\n                    for tool_call in tool_calls:\n                        if not isinstance(tool_call, dict):\n                            continue\n                        function = tool_call.get("function")\n                        if not isinstance(function, dict):\n                            continue\n                        value = function.get("arguments")\n                        if isinstance(value, str) and value.strip():\n                            arguments.append(value.strip())\n                        elif isinstance(value, dict):\n                            arguments.append(json.dumps(value))\n                    if arguments:\n                        return "\\n".join(arguments)\n\n                function_call = message.get("function_call")\n                if isinstance(function_call, dict):\n                    arguments = function_call.get("arguments")\n                    if isinstance(arguments, str) and arguments.strip():\n                        return arguments.strip()\n                    if isinstance(arguments, dict):\n                        return json.dumps(arguments)\n\n            completion_text = text_from(choice.get("text"))\n            if completion_text:\n                return completion_text\n\n        output = text_from(body.get("output"))\n        if output:\n            return output\n        response = text_from(body.get("response"))\n        if response:\n            return response\n        raise ModelProtocolError("unsupported or empty model response schema")\n\n    @staticmethod\n    def _parse_json(content: str) -> dict[str, Any]:\n        cleaned = re.sub(r"<think>.*?</think>", "", str(content), flags=re.DOTALL | re.IGNORECASE)\n        cleaned = cleaned.replace("```json", "").replace("```", "").strip()\n        decoder = json.JSONDecoder()\n        for index, char in enumerate(cleaned):\n            if char != "{":\n                continue\n            try:\n                value, _end = decoder.raw_decode(cleaned[index:])\n            except json.JSONDecodeError:\n                continue\n            if isinstance(value, dict):\n                return value\n        raise ModelProtocolError("model response did not contain a JSON object")\n\n    def _validate_decision(\n        self,\n        parsed: dict[str, Any],\n        candidates: list[ComboCandidate],\n        available: set[str],\n    ) -> ModelDecision:\n        raw_index = parsed.get("selected_candidate", 0)\n        try:\n            selected_index = int(raw_index)\n        except (TypeError, ValueError):\n            selected_index = 0\n        if not 0 <= selected_index < len(candidates):\n            selected_index = 0\n        selected = candidates[selected_index].actions\n        if any(action.name not in available for action in selected):\n            selected = candidates[0].actions\n        world_model = clip_text(parsed.get("world_model"), 700) or self.world.world_model_text\n        goal_model = clip_text(parsed.get("goal_model"), 500) or self.world.goal_model_text\n        reasoning = clip_text(parsed.get("reasoning_summary"), 500) or candidates[selected_index].rationale\n        self.world.world_model_text = world_model\n        self.world.goal_model_text = goal_model\n        return ModelDecision(\n            selected=selected,\n            world_model=world_model,\n            goal_model=goal_model,\n            reasoning_summary=reasoning,\n            source="local-model",\n        )\n\n    def _heuristic(self, candidates: list[ComboCandidate], reason: str) -> ModelDecision:\n        candidate = candidates[0]\n        return ModelDecision(\n            selected=candidate.actions,\n            world_model=self.world.world_model_text,\n            goal_model=self.world.goal_model_text,\n            reasoning_summary=f"{reason}; selected highest evidence score: {candidate.rationale}",\n            source="heuristic",\n        )\n\n\nclass MyAgent(Agent):\n    """Object-centric, model-assisted ARC-AGI-3 competition agent."""\n\n    MAX_ACTIONS = max(80, int(os.environ.get("ARC3_MAX_ACTIONS", "320")))\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self.world = GameWorldModel(self.game_id)\n        self.planner = LocalModelPlanner(self.game_id, self.world)\n        self.plan_queue: deque[ActionSpec] = deque()\n        self.plan_id = 0\n        self.pending_action = ActionSpec("RESET")\n        self.pending_reasoning: dict[str, Any] = {}\n        self.last_observation: Optional[FrameView] = None\n        self.last_evidence: Optional[TransitionEvidence] = None\n        self.plan_source = "bootstrap"\n        legal = [normalize_action_name(action) for action in getattr(self.arc_env, "action_space", [])]\n        log(\n            self.game_id,\n            "LOAD",\n            f"environment loaded; legal={legal or \'from first observation\'}; max_actions={self.MAX_ACTIONS}; "\n            f"planner={MODEL_ENDPOINT}",\n        )\n\n    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:\n        return latest_frame.state is GameState.WIN or self.action_counter >= self.MAX_ACTIONS\n\n    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:\n        view = analyze_frame(latest_frame, self.action_counter)\n        self.last_observation = view\n        available = self._available_action_names(latest_frame)\n        object_preview = " | ".join(component.compact() for component in view.components[:10])\n        log(\n            self.game_id,\n            "OBSERVE",\n            f"step={self.action_counter} state={view.state_name} levels={view.level} sig={view.signature} "\n            f"bg={view.background} colors={view.color_counts} objects={len(view.components)}",\n        )\n        log(self.game_id, "OBJECTS", object_preview or "no non-background connected components")\n\n        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):\n            self.plan_queue.clear()\n            reason = "start the environment" if latest_frame.state is GameState.NOT_PLAYED else "recover current level after GAME_OVER"\n            return self._commit(ActionSpec("RESET"), source="protocol", reason=reason)\n\n        if view.level != self.world.last_level:\n            self.plan_queue.clear()\n            log(self.game_id, "LEVEL", f"new settled level detected: completed={view.level}; discarded stale queued moves")\n            self.world.last_level = view.level\n\n        queue_is_safe = (\n            bool(self.plan_queue)\n            and self.last_evidence is not None\n            and self.last_evidence.changed_cells > 0\n            and not self.last_evidence.level_completed\n            and self.last_evidence.state_after not in {"GAME_OVER", "WIN"}\n        )\n        if not queue_is_safe:\n            self.plan_queue.clear()\n\n        if not self.plan_queue:\n            candidates = self.world.generate_candidates(view, available)\n            if not candidates:\n                legal = sorted(name for name in available if name != "RESET")\n                if not legal:\n                    return self._commit(ActionSpec("RESET"), source="protocol", reason="no non-reset action advertised")\n                candidates = [self.world.score_combo(view, (ActionSpec(legal[0]),))]\n            log(\n                self.game_id,\n                "CANDIDATES",\n                " || ".join(f"#{index} {candidate.compact()}" for index, candidate in enumerate(candidates[:6])),\n            )\n            decision = self.planner.choose(view, candidates, available)\n            self.plan_id += 1\n            self.plan_source = decision.source\n            self.plan_queue.extend(decision.selected)\n            selected_key = ">".join(action.key for action in decision.selected)\n            self.world.combo_visits[selected_key] += 1\n            log(self.game_id, "WORLD", decision.world_model)\n            log(self.game_id, "GOAL", decision.goal_model)\n            log(\n                self.game_id,\n                "REASON",\n                f"plan={self.plan_id} source={decision.source} selected={selected_key}; {decision.reasoning_summary}",\n            )\n\n        action = self.plan_queue.popleft()\n        if action.name not in available:\n            self.plan_queue.clear()\n            legal = sorted(name for name in available if name != "RESET")\n            action = ActionSpec(legal[0]) if legal else ActionSpec("RESET")\n        return self._commit(\n            action,\n            source=self.plan_source,\n            reason=f"execute plan {self.plan_id}; {len(self.plan_queue)} queued move(s) remain",\n        )\n\n    def _commit(self, action: ActionSpec, source: str, reason: str) -> GameAction:\n        self.pending_action = action\n        self.pending_reasoning = {\n            "plan_id": self.plan_id,\n            "source": source,\n            "summary": clip_text(reason, 600),\n            "world_model": clip_text(self.world.world_model_text, 1000),\n            "goal_model": clip_text(self.world.goal_model_text, 700),\n        }\n        log(\n            self.game_id,\n            "EXECUTE",\n            f"actual_move={action.name} data={action.data} plan={self.plan_id} source={source}; {reason}",\n        )\n        try:\n            return GameAction.from_name(action.name)\n        except Exception:\n            return getattr(GameAction, action.name)\n\n    def do_action_request(self, action: GameAction) -> FrameData:\n        raw = self.arc_env.step(\n            action,\n            data=dict(self.pending_action.data),\n            reasoning=dict(self.pending_reasoning),\n        )\n        return self._convert_raw_frame_data(raw)\n\n    def append_frame(self, frame: FrameData) -> None:\n        after = analyze_frame(frame, self.action_counter + 1)\n        if self.last_observation is not None:\n            evidence = self.world.record_transition(self.last_observation, after, self.pending_action)\n            self.last_evidence = evidence\n            log(\n                self.game_id,\n                "RESULT",\n                f"move={self.pending_action.key} changed_cells={evidence.changed_cells} bbox={evidence.changed_bbox} "\n                f"color_delta={list(evidence.color_delta)} moved={list(evidence.moved[:5])} "\n                f"state={evidence.state_after} levels={evidence.level_before}->{evidence.level_after}",\n            )\n            if (\n                evidence.changed_cells == 0\n                or evidence.level_completed\n                or evidence.state_after in {"GAME_OVER", "WIN"}\n            ):\n                if self.plan_queue:\n                    log(\n                        self.game_id,\n                        "REPLAN",\n                        f"discarding {len(self.plan_queue)} queued move(s) after boundary/no-op/terminal evidence",\n                    )\n                self.plan_queue.clear()\n        self.last_observation = after\n        super().append_frame(frame)\n\n    def _available_action_names(self, latest_frame: FrameData) -> set[str]:\n        names: set[str] = set()\n        for value in getattr(latest_frame, "available_actions", []) or []:\n            try:\n                if isinstance(value, GameAction):\n                    names.add(value.name)\n                else:\n                    names.add(GameAction.from_id(int(value)).name)\n            except Exception:\n                normalized = normalize_action_name(value)\n                if normalized:\n                    names.add(normalized)\n        if not names:\n            for action in getattr(self.arc_env, "action_space", []) or []:\n                names.add(normalize_action_name(action))\n        names.add("RESET")\n        return names\n'
Path('/tmp/my_agent.py').write_text(AGENT_SOURCE, encoding='utf-8')
Path('/kaggle/working/agent.py').write_text(AGENT_SOURCE, encoding='utf-8')
compile(AGENT_SOURCE, '/tmp/my_agent.py', 'exec')
print(f'[AGENT] wrote /tmp/my_agent.py and /kaggle/working/agent.py ({len(AGENT_SOURCE.splitlines())} lines)')


## 3. Start the local reasoning model only for the hidden scored rerun

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def find_vllm_wheelhouse() -> Path:
    candidates = []
    for wheel in Path('/kaggle/input').rglob('vllm*.whl'):
        text = str(wheel.parent).lower()
        score = 0
        score += 10 if 'wheelhouse' in text else 0
        score += 5 if 'h100' in text or 'cuda' in text else 0
        candidates.append((score, wheel.parent))
    if not candidates:
        raise FileNotFoundError('No attached vLLM wheelhouse was found under /kaggle/input.')
    return max(candidates, key=lambda item: item[0])[1]


def model_score(path: Path, config: dict) -> int:
    text = (str(path) + ' ' + json.dumps(config)).lower()
    score = 0
    score += 80 if 'qwen3.6' in text else 0
    score += 50 if '27b' in text else 0
    score += 35 if 'fp8' in text else 0
    score += 10 if 'qwen' in text else 0
    score -= 100 if any(word in text for word in ('embedding', 'reranker', 'tokenizer-only')) else 0
    return score


def find_model_snapshot() -> Path:
    candidates = []
    for config_path in Path('/kaggle/input').rglob('config.json'):
        model_dir = config_path.parent
        if str(model_dir).startswith(str(COMP_ROOT)):
            continue
        has_weights = any(model_dir.glob('*.safetensors')) or any(model_dir.glob('*.bin'))
        has_index = any(model_dir.glob('*.index.json'))
        if not (has_weights or has_index):
            continue
        try:
            config = json.loads(config_path.read_text(encoding='utf-8'))
        except Exception:
            continue
        candidates.append((model_score(model_dir, config), model_dir))
    if not candidates:
        raise FileNotFoundError('No attached local language-model snapshot with weights was found.')
    score, path = max(candidates, key=lambda item: item[0])
    if score < 50:
        raise RuntimeError(f'Attached models do not include the expected Qwen 3.6 27B FP8 snapshot; best={path}.')
    return path


def wait_for_url(url: str, timeout_s: float, label: str) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ''
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=10) as response:
                if response.status < 500:
                    print(f'[{label}] ready: {url}', flush=True)
                    return
        except Exception as exc:
            last_error = f'{type(exc).__name__}: {exc}'
        time.sleep(5)
    raise RuntimeError(f'{label} did not become ready within {timeout_s:.0f}s: {last_error}')


def validate_chat_schema(url: str) -> None:
    payload = {
        'model': 'arc3-local',
        'messages': [
            {'role': 'system', 'content': 'Return only strict JSON.'},
            {'role': 'user', 'content': '{"selected_candidate":0}'},
        ],
        'temperature': 0,
        'max_tokens': 64,
        'stream': False,
    }
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'Authorization': 'Bearer arc3-local'},
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        raw_body = response.read().decode('utf-8')
    try:
        body = json.loads(raw_body)
        choices = body['choices']
        message = choices[0]['message']
    except (json.JSONDecodeError, KeyError, IndexError, TypeError) as exc:
        raise RuntimeError('Model endpoint is not chat-completions compatible.') from exc
    content = message.get('content') if isinstance(message, dict) else None
    tool_calls = message.get('tool_calls') if isinstance(message, dict) else None
    if not ((isinstance(content, str) and content.strip()) or (isinstance(tool_calls, list) and tool_calls)):
        raise RuntimeError('Model endpoint returned neither message content nor tool-call arguments.')
    print(f'[MODEL] chat schema validated: choices={len(choices)}', flush=True)


VLLM_PROCESS = None
MODEL_PROTOCOL_OK = False
if TRUE_SUBMISSION:
    try:
        import vllm  # noqa: F401
        print('[MODEL] vLLM is already installed')
    except Exception:
        wheelhouse = find_vllm_wheelhouse()
        print(f'[MODEL] installing vLLM from {wheelhouse}', flush=True)
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
            '--disable-pip-version-check', '--find-links', str(wheelhouse), 'vllm',
        ])

    model_path = find_model_snapshot()
    print(f'[MODEL] selected snapshot: {model_path}', flush=True)
    model_log = (WORKING / 'vllm.log').open('w', encoding='utf-8')
    server_env = os.environ.copy()
    server_env.update({
        'HF_HUB_OFFLINE': '1',
        'TRANSFORMERS_OFFLINE': '1',
        'TOKENIZERS_PARALLELISM': 'true',
        'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    })
    command = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', str(model_path),
        '--served-model-name', 'arc3-local',
        '--host', '127.0.0.1',
        '--port', '1234',
        '--tensor-parallel-size', '1',
        '--dtype', 'auto',
        '--gpu-memory-utilization', '0.92',
        '--max-model-len', '32768',
        '--max-num-seqs', '16',
        '--enable-prefix-caching',
        '--trust-remote-code',
        '--generation-config', 'vllm',
    ]
    print('[MODEL] launching: ' + ' '.join(command), flush=True)
    VLLM_PROCESS = subprocess.Popen(command, stdout=model_log, stderr=subprocess.STDOUT, env=server_env)
    try:
        wait_for_url('http://127.0.0.1:1234/v1/models', 1200, 'MODEL')
    except Exception:
        model_log.flush()
        tail = (WORKING / 'vllm.log').read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
        print('[MODEL] startup log tail:\n' + '\n'.join(tail), flush=True)
        raise
    try:
        validate_chat_schema('http://127.0.0.1:1234/v1/chat/completions')
        MODEL_PROTOCOL_OK = True
    except Exception as exc:
        print(
            f'[MODEL] preflight failed; disabling model planner before environment actions: '
            f'{type(exc).__name__}: {exc}',
            flush=True,
        )
else:
    print('[MODEL] validation mode: model startup intentionally skipped')


## 4. Run through the official Kaggle gateway and validate the artifact

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import pandas as pd


def wait_for_gateway(timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ''
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen('http://gateway:8001/api/games', timeout=10) as response:
                if response.status < 500:
                    games = json.loads(response.read().decode('utf-8'))
                    print(f'[GATEWAY] ready; environments={len(games)}', flush=True)
                    return
        except Exception as exc:
            last_error = f'{type(exc).__name__}: {exc}'
        time.sleep(5)
    raise RuntimeError(f'Kaggle gateway did not become ready: {last_error}')


SUBMISSION_PATH = WORKING / 'submission.parquet'
if TRUE_SUBMISSION:
    source_repo = COMP_ROOT / 'ARC-AGI-3-Agents'
    runtime_repo = WORKING / 'ARC-AGI-3-Agents'
    wait_for_gateway()
    if runtime_repo.exists():
        shutil.rmtree(runtime_repo)
    shutil.copytree(source_repo, runtime_repo)
    shutil.copy2('/tmp/my_agent.py', runtime_repo / 'agents/templates/my_agent.py')

    registry_source = """from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}

__all__ = ["Agent", "Playback", "Swarm", "Random", "MyAgent", "AVAILABLE_AGENTS"]
"""
    (runtime_repo / 'agents/__init__.py').write_text(registry_source, encoding='utf-8')
    (runtime_repo / '.env').write_text(
        'SCHEME=http\n'
        'HOST=gateway\n'
        'PORT=8001\n'
        'ARC_API_KEY=test-key-123\n'
        'ARC_BASE_URL=http://gateway:8001/\n'
        'OPERATION_MODE=online\n'
        'ENVIRONMENTS_DIR=\n'
        'RECORDINGS_DIR=/kaggle/working/server_recording\n',
        encoding='utf-8',
    )
    run_env = os.environ.copy()
    run_env.update({
        'ARC3_MODEL_ENDPOINT': 'http://127.0.0.1:1234/v1/chat/completions',
        'ARC3_MODEL_NAME': 'arc3-local',
        'ARC3_MODEL_CONCURRENCY': '12',
        'ARC3_MODEL_TIMEOUT_S': '240',
        'ARC3_DISABLE_MODEL': '0' if MODEL_PROTOCOL_OK else '1',
        'ARC3_MAX_ACTIONS': '320',
        'ARC3_MAX_PLAN_LENGTH': '8',
        'PYTHONUNBUFFERED': '1',
    })
    print('[RUN] starting official main.py --agent myagent; detailed traces follow', flush=True)
    try:
        subprocess.run(
            [sys.executable, 'main.py', '--agent', 'myagent'],
            cwd=runtime_repo,
            env=run_env,
            check=True,
        )
    finally:
        if VLLM_PROCESS is not None and VLLM_PROCESS.poll() is None:
            VLLM_PROCESS.terminate()
            try:
                VLLM_PROCESS.wait(timeout=30)
            except subprocess.TimeoutExpired:
                VLLM_PROCESS.kill()
else:
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    ).to_parquet(SUBMISSION_PATH, index=False)
    print('[RUN] validation mode: wrote canonical gate parquet; no game or training run was executed')

if not SUBMISSION_PATH.is_file():
    raise FileNotFoundError(f'Required artifact was not produced: {SUBMISSION_PATH}')
submission = pd.read_parquet(SUBMISSION_PATH)
required_columns = ['row_id', 'game_id', 'end_of_game', 'score']
if list(submission.columns) != required_columns:
    raise ValueError(f'Wrong submission schema: {list(submission.columns)} != {required_columns}')
if submission.empty:
    raise ValueError('submission.parquet is empty')
print(f'[ARTIFACT] {SUBMISSION_PATH} rows={len(submission)} bytes={SUBMISSION_PATH.stat().st_size}')
print(submission.head(20).to_string(index=False))
